# Real-Time Exercise Form Analysis — Milestone Demo
**CS [Course] | Zaid Akhtar & Heena Khan**

This notebook demonstrates the core pipeline:
1. BlazePose landmark extraction on a squat video
2. Joint angle computation via vector dot products
3. EMA temporal smoothing
4. Color-coded skeleton overlay with corrective cues
5. Rep counting via angle-threshold crossing

In [2]:
# Install dependencies
!pip install mediapipe opencv-python-headless matplotlib numpy -q
!apt-get install -y ffmpeg -q  # for video decoding


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
zsh:1: command not found: apt-get


In [6]:
from google.colab import files

uploaded = files.upload()

ModuleNotFoundError: No module named 'google.colab'

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from google.colab.patches import cv2_imshow
from IPython.display import display
import urllib.request

# MediaPipe setup
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print('MediaPipe version:', mp.__version__)
print('OpenCV version:', cv2.__version__)

ModuleNotFoundError: No module named 'google.colab'

In [4]:
# ============================================================
# CORE FUNCTIONS
# ============================================================

def compute_angle(a, b, c):
    """
    Compute the angle at joint b given three landmark points a, b, c.
    Uses vector dot product: cos(theta) = (ba . bc) / (|ba| * |bc|)
    Returns angle in degrees.
    """
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))


class EMAFilter:
    """
    Exponential Moving Average filter for temporal smoothing of joint angles.
    Weights recent frames more heavily: s_t = alpha * x_t + (1-alpha) * s_{t-1}
    """
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.value = None

    def update(self, x):
        if self.value is None:
            self.value = x
        else:
            self.value = self.alpha * x + (1 - self.alpha) * self.value
        return self.value


class RepCounter:
    """
    Counts reps via angle-threshold crossing.
    A rep is completed when angle drops below down_thresh then rises above up_thresh.
    """
    def __init__(self, down_thresh=110, up_thresh=150):
        self.down_thresh = down_thresh
        self.up_thresh = up_thresh
        self.stage = 'up'  # 'up' or 'down'
        self.count = 0

    def update(self, angle):
        if angle < self.down_thresh:
            self.stage = 'down'
        if angle > self.up_thresh and self.stage == 'down':
            self.stage = 'up'
            self.count += 1
        return self.count, self.stage


# Squat form thresholds
THRESHOLDS = {
    'knee_angle_min': 90,   # below = good depth
    'knee_angle_max': 100,  # above at bottom = not deep enough
    'hip_angle_min': 50,    # forward lean check
}

def get_squat_cues(knee_angle, hip_angle=None):
    """Return corrective cue string and color based on angles."""
    if knee_angle > 150:
        return 'DESCENDING...', (255, 255, 255)
    elif knee_angle > THRESHOLDS['knee_angle_max']:
        return 'GO DEEPER - bend knees more', (0, 100, 255)   # orange in BGR
    elif knee_angle < THRESHOLDS['knee_angle_min']:
        return 'GOOD DEPTH', (0, 200, 50)                     # green
    else:
        return 'APPROACHING DEPTH', (0, 200, 200)             # yellow

print('Core functions defined.')

Core functions defined.


In [5]:
# ============================================================
# DOWNLOAD PUBLIC SQUAT VIDEO
# Using a Creative Commons squat clip from Wikimedia Commons
# ============================================================

VIDEO_URL = 'https://upload.wikimedia.org/wikipedia/commons/transcoded/6/6a/Squat_side_view.webm/Squat_side_view.webm.480p.vp9.webm'
VIDEO_PATH = 'squat_sample.webm'

print('Downloading squat video...')
urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
print('Downloaded.')

NameError: name 'urllib' is not defined

In [ ]:
# ============================================================
# RUN BLAZEPOSE + ANGLE EXTRACTION ON VIDEO
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {total_frames} frames @ {fps:.1f} fps')

# Storage
raw_knee_angles = []
smoothed_knee_angles = []
frame_indices = []
annotated_frames = []  # save select frames for display

ema = EMAFilter(alpha=0.3)
rep_counter = RepCounter(down_thresh=110, up_thresh=150)

LANDMARK = mp_pose.PoseLandmark

with mp_pose.Pose(min_detection_confidence=0.5,
                  min_tracking_confidence=0.5,
                  model_complexity=1) as pose:

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR -> RGB for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = pose.process(rgb)
        rgb.flags.writeable = True
        frame = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark
            h, w, _ = frame.shape

            def get_xy(lm_id):
                p = lm[lm_id]
                return [p.x * w, p.y * h]

            # Right side landmarks
            r_hip   = get_xy(LANDMARK.RIGHT_HIP)
            r_knee  = get_xy(LANDMARK.RIGHT_KNEE)
            r_ankle = get_xy(LANDMARK.RIGHT_ANKLE)

            # Compute knee flexion angle
            raw_angle = compute_angle(r_hip, r_knee, r_ankle)
            smooth_angle = ema.update(raw_angle)

            raw_knee_angles.append(raw_angle)
            smoothed_knee_angles.append(smooth_angle)
            frame_indices.append(frame_idx)

            # Rep counting
            reps, stage = rep_counter.update(smooth_angle)

            # Get cue
            cue_text, cue_color = get_squat_cues(smooth_angle)

            # Draw skeleton
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
            )

            # Annotate knee angle
            knee_px = tuple(map(int, r_knee))
            angle_color = (0, 200, 50) if smooth_angle < 100 else (0, 100, 255)
            cv2.putText(frame, f'{smooth_angle:.0f}deg',
                        (knee_px[0]+10, knee_px[1]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, angle_color, 2)

            # Corrective cue banner
            cv2.rectangle(frame, (0, 0), (w, 50), (30, 30, 30), -1)
            cv2.putText(frame, cue_text, (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, cue_color, 2)

            # Rep counter
            cv2.putText(frame, f'REPS: {reps}  [{stage.upper()}]',
                        (w - 220, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            # Save select frames for display
            if frame_idx % max(1, total_frames // 6) == 0:
                annotated_frames.append(frame.copy())

        frame_idx += 1

cap.release()
print(f'Processed {len(raw_knee_angles)} frames with detected pose.')
print(f'Total reps counted: {rep_counter.count}')

In [ ]:
# ============================================================
# VISUALIZATION 1: Angle Time Series (Raw vs EMA)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a2e')

ax = axes[0]
ax.plot(frame_indices, raw_knee_angles, color='#888888', alpha=0.5, lw=1, label='Raw')
ax.plot(frame_indices, smoothed_knee_angles, color='#00d4ff', lw=2.5, label='EMA Smoothed')
ax.axhline(100, color='#ff4444', lw=1.5, ls='--', label='Form threshold (100°)')
ax.axhline(90, color='#ff8800', lw=1.5, ls=':', label='Depth target (90°)')
ax.fill_between(frame_indices, smoothed_knee_angles, 100,
                where=[a > 100 for a in smoothed_knee_angles],
                alpha=0.2, color='#ff4444')
ax.set_xlabel('Frame', color='white'); ax.set_ylabel('Knee Flexion Angle (°)', color='white')
ax.set_title('Knee Angle — Raw vs EMA Smoothed', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#333', fontsize=8)
for s in ax.spines.values(): s.set_edgecolor('#333')

# Angle distribution histogram
ax2 = axes[1]
ax2.hist(smoothed_knee_angles, bins=30, color='#00d4ff', alpha=0.75, edgecolor='#0f0f0f')
ax2.axvline(90, color='#44ff88', lw=2, ls='--', label='Target depth (90°)')
ax2.axvline(100, color='#ff4444', lw=2, ls='--', label='Form threshold (100°)')
ax2.set_xlabel('Knee Angle (°)', color='white'); ax2.set_ylabel('Frame Count', color='white')
ax2.set_title('Angle Distribution Across Session', color='white', fontweight='bold')
ax2.tick_params(colors='white')
ax2.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#333', fontsize=8)
for s in ax2.spines.values(): s.set_edgecolor('#333')

plt.tight_layout()
plt.savefig('angle_timeseries_real.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION 2: Show Annotated Frames
# ============================================================

if annotated_frames:
    n = min(3, len(annotated_frames))
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1: axes = [axes]
    for i, (ax, frame) in enumerate(zip(axes, annotated_frames[:n])):
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {i+1}', color='white')
        ax.axis('off')
    fig.patch.set_facecolor('#0f0f0f')
    plt.suptitle('BlazePose Skeleton Overlay with Angle Annotations', color='white', fontsize=13)
    plt.tight_layout()
    plt.savefig('skeleton_overlay_real.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
else:
    print('No annotated frames available (pose not detected in video). Check video format.')

In [ ]:
# ============================================================
# EVALUATION STUB: Precision / Recall on form errors
# (Will be populated fully in Week 10 with InfiniteRep labels)
# ============================================================

def evaluate_form_detection(predicted_flags, ground_truth_flags):
    """
    Compute precision and recall for form error detection.
    predicted_flags: list of booleans (True = error flagged by system)
    ground_truth_flags: list of booleans (True = actual error frame)
    """
    tp = sum(p and g for p, g in zip(predicted_flags, ground_truth_flags))
    fp = sum(p and not g for p, g in zip(predicted_flags, ground_truth_flags))
    fn = sum(not p and g for p, g in zip(predicted_flags, ground_truth_flags))
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    return {'precision': precision, 'recall': recall, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}

# Example with synthetic labels (to be replaced with InfiniteRep ground truth)
n = len(smoothed_knee_angles)
predicted = [a > 100 for a in smoothed_knee_angles]  # flag if not deep enough
# Synthetic GT: assume middle third of frames should be flagged
gt = [i > n//3 and i < 2*n//3 for i in range(n)]
metrics = evaluate_form_detection(predicted, gt)
print('Evaluation (synthetic GT placeholder):')
for k, v in metrics.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')
print('\nNote: real evaluation against InfiniteRep labels scheduled for Week 10.')